# 官公庁AI市場分析用 LLM特徴量生成（2_01）

行政事業の4テキスト列を固定taxonomyへ分類し、`policy_domain`、`admin_process`、`ai_usecase`、`ai_applicability`を生成します。LLMには市場性・AIU fit・期待効果を評価させません。

処理順は **全件dry-run → 10件sample → 人手確認 → 全件resume実行 → CSV保存 → 分布確認** です。API実行フラグは初期状態ですべて`False`です。

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from llm_features import (
    ADMIN_PROCESSES,
    AI_USECASES,
    POLICY_DOMAINS,
    PROMPT_VERSION,
    decode_ai_usecase,
    filter_projects_by_start_year,
    generate_llm_features,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 160)

## 1. train/testを同じ行政事業テーブルとして準備

train/testそれぞれを`project_start_year >= 2020`へ限定します。元の`project_id`、年度、splitを保持し、API処理用には`train::ID` / `test::ID`という一意キーを作ります。年度、目的変数、省庁、予算等の構造化列はLLMへ送りません。

In [ ]:
ID_COL = 'project_id'
YEAR_COL = 'project_start_year'
MIN_START_YEAR = 2020
LLM_ID_COL = 'llm_row_id'
TEXT_COLS = ['project_name', 'project_objective', 'project_summary', 'current_issues']

train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')
train_eligible = filter_projects_by_start_year(train, year_col=YEAR_COL, min_year=MIN_START_YEAR)
test_eligible = filter_projects_by_start_year(test, year_col=YEAR_COL, min_year=MIN_START_YEAR)

def prepare_projects(frame, source_split):
    missing = [column for column in [ID_COL, YEAR_COL, *TEXT_COLS] if column not in frame.columns]
    if missing:
        raise KeyError(f'{source_split}に必要列がありません: {missing}')
    result = frame[[ID_COL, YEAR_COL, *TEXT_COLS]].copy()
    result['source_split'] = source_split
    result['source_index'] = frame.index
    result[LLM_ID_COL] = source_split + '::' + result[ID_COL].astype(str)
    return result

projects = pd.concat(
    [prepare_projects(train_eligible, 'train'), prepare_projects(test_eligible, 'test')],
    ignore_index=True,
)
assert projects[LLM_ID_COL].notna().all() and projects[LLM_ID_COL].is_unique
print('original train/test:', len(train), len(test))
print(f'project_start_year >= {MIN_START_YEAR}:', len(train_eligible), len(test_eligible), len(projects))
display(projects.head())

## 2. Bedrock・料金・安全設定

既定候補は低コストのAmazon Nova Microです。AWS公式model card上、Nova Microは`Converse`とtool useに対応しますが、Tokyoでのin-region提供はありません。このNotebookは`us-east-1`を既定にしているため、データのリージョン要件がある場合は実行前に利用可能な別モデルへ変更してください。

下記単価は2026-08-26にAWS公式情報で確認した参考値です。**実行当日にBedrock Pricingを再確認して更新してください。** `MAX_BUDGET_USD=20`は見積り時と実行中の両方で検査されます。

モデル変更は`MODEL_ID`だけで済むとは限りません。少なくとも`REGION_NAME`、入力・出力単価、model access、Converseのtool use/toolChoice対応を確認します。`ADDITIONAL_MODEL_REQUEST_FIELDS`の`topK`はNova向けなので、他モデルでは`None`またはそのモデル固有設定へ変更してください。

In [ ]:
MODEL_ID = 'amazon.nova-micro-v1:0'
REGION_NAME = 'us-east-1'
AWS_PROFILE_NAME = os.getenv('AWS_PROFILE')  # AWS上では通常None
INPUT_PRICE_PER_MILLION = 0.035   # 実行日に再確認
OUTPUT_PRICE_PER_MILLION = 0.14   # 実行日に再確認
MAX_BUDGET_USD = 20.0
MAX_OUTPUT_TOKENS = 400
MAX_CHARACTERS_PER_FIELD = 4000
CHARACTERS_PER_TOKEN = 1.0  # 日本語に対して保守的に見積もる
SAMPLE_N_ROWS = 10
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'llm_features'
CANONICAL_OUTPUT = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_llm_features.csv.gz'

RUN_SAMPLE_API = False
RUN_FULL_API = False
SAVE_RAW_RESPONSE = False
ADDITIONAL_MODEL_REQUEST_FIELDS = {'inferenceConfig': {'topK': 1}}  # Nova向け

COMMON_ARGS = dict(
    split='projects_start_year_2020_plus',
    model=MODEL_ID,
    output_root=OUTPUT_ROOT,
    text_cols=TEXT_COLS,
    project_id_col=LLM_ID_COL,
    prompt_version=PROMPT_VERSION,
    region_name=REGION_NAME,
    aws_profile_name=AWS_PROFILE_NAME,
    max_characters_per_field=MAX_CHARACTERS_PER_FIELD,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    characters_per_token=CHARACTERS_PER_TOKEN,
    input_price_per_million=INPUT_PRICE_PER_MILLION,
    output_price_per_million=OUTPUT_PRICE_PER_MILLION,
    max_budget_usd=MAX_BUDGET_USD,
    max_retries=3,
    save_raw_response=SAVE_RAW_RESPONSE,
    additional_model_request_fields=ADDITIONAL_MODEL_REQUEST_FIELDS,
)

## 3. 全件dry-run（API呼び出しなし）

対象件数、最大想定token、最大想定費用、実際のsample promptを確認します。出力費用は各行が`MAX_OUTPUT_TOKENS`をすべて使う保守的な上限です。

In [ ]:
dry_run = generate_llm_features(projects, dry_run=True, **COMMON_ARGS)
dry_report = pd.Series({k: v for k, v in dry_run['report'].items() if k != 'sample_prompt'})
display(dry_report.to_frame('value'))
print(dry_run['report']['sample_prompt'][:5000])
if dry_run['report']['estimated_max_cost_usd'] > MAX_BUDGET_USD:
    raise RuntimeError('全件の最大見積りが20ドルを超えています。文字数・対象件数・モデルを見直してください。')

## 4. まず10件だけ実行して人間が確認

`RUN_SAMPLE_API=True`にした場合だけ課金APIを呼びます。成功した各行は即時checkpointされ、同じ設定で全件実行すると再利用されます。Access keyはNotebookへ書かず、boto3標準credential chainを使います。

In [ ]:
sample_result = None
if RUN_SAMPLE_API:
    sample_result = generate_llm_features(
        projects, n_rows=SAMPLE_N_ROWS, dry_run=False, **COMMON_ARGS
    )
    sample_features = sample_result['features'].rename(columns={'project_id': LLM_ID_COL})
    sample_source = projects.reset_index(names='row_position')
    review = sample_source.merge(sample_features, on=['row_position', LLM_ID_COL], how='inner')
    display(review[[
        ID_COL, YEAR_COL, *TEXT_COLS, 'policy_domain', 'admin_process', 'ai_usecase',
        'ai_applicability', 'classification_reason',
    ]])
    display(sample_result['errors'])
    display(pd.Series(sample_result['report']).to_frame('value'))
else:
    print('Sample API is disabled. 単価・region・promptを確認後、RUN_SAMPLE_API=Trueにしてください。')

## 5. 全件生成・元ID付きCSV保存

sampleの分類品質を確認してから`RUN_FULL_API=True`にします。中断時も成功済み行は再課金せず、未処理・失敗行から再開します。不完全な結果はcanonical CSVへ保存しません。

In [ ]:
full_result = None
canonical_features = None
if RUN_FULL_API:
    full_result = generate_llm_features(projects, n_rows=None, dry_run=False, **COMMON_ARGS)
    if not full_result['errors'].empty or len(full_result['features']) != len(projects):
        display(full_result['errors'])
        raise RuntimeError('未分類行があります。errorを確認し、同じ設定で再実行してください。')

    generated = full_result['features'].rename(columns={'project_id': LLM_ID_COL})
    source_keys = projects.reset_index(names='row_position')[[
        'row_position', LLM_ID_COL, ID_COL, YEAR_COL, 'source_split', 'source_index'
    ]]
    canonical_features = source_keys.merge(
        generated, on=['row_position', LLM_ID_COL], how='left', validate='one_to_one'
    )
    assert canonical_features['policy_domain'].notna().all()
    CANONICAL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    canonical_features.to_csv(CANONICAL_OUTPUT, index=False, compression='gzip')
    print('saved:', CANONICAL_OUTPUT, 'rows:', len(canonical_features))
    display(canonical_features.head())
else:
    print('Full API is disabled. Sample確認後にRUN_FULL_API=Trueへ変更してください。')

## 6. 保存済み特徴量の分布

全件CSVがあれば読み込みます。なければ、このsessionのsample結果を可視化対象にします。

In [ ]:
analysis_features = canonical_features
if analysis_features is None and CANONICAL_OUTPUT.exists():
    analysis_features = pd.read_csv(CANONICAL_OUTPUT, compression='gzip')
elif analysis_features is None and sample_result is not None:
    analysis_features = sample_result['features']

if analysis_features is None:
    print('可視化対象がありません。sampleまたはfullを実行してください。')
else:
    plot_frame = analysis_features.copy()
    plot_frame['policy_domain_label'] = plot_frame['policy_domain'].map(
        lambda code: f"{code} {POLICY_DOMAINS.get(code, '不明')}"
    )
    plot_frame['admin_process_label'] = plot_frame['admin_process'].map(
        lambda code: f"{code} {ADMIN_PROCESSES.get(code, '不明')}"
    )
    fig, axes = plt.subplots(3, 1, figsize=(14, 18))
    sns.countplot(data=plot_frame, y='policy_domain_label', order=plot_frame['policy_domain_label'].value_counts().index, ax=axes[0])
    sns.countplot(data=plot_frame, y='admin_process_label', order=plot_frame['admin_process_label'].value_counts().index, ax=axes[1])
    sns.countplot(data=plot_frame, x='ai_applicability', order=[0, 1, 2], ax=axes[2])
    axes[0].set_title('policy_domain 件数')
    axes[1].set_title('admin_process 件数')
    axes[2].set_title('ai_applicability 件数')
    plt.tight_layout()
    plt.show()

    exploded = plot_frame.assign(
        ai_usecase=plot_frame['ai_usecase'].map(decode_ai_usecase)
    ).explode('ai_usecase')
    exploded['ai_usecase_label'] = exploded['ai_usecase'].map(
        lambda code: f"{code} {AI_USECASES.get(code, '不明')}"
    )
    plt.figure(figsize=(14, 7))
    sns.countplot(data=exploded, y='ai_usecase_label', order=exploded['ai_usecase_label'].value_counts().index)
    plt.title('ai_usecase 件数（multi-label explode）')
    plt.tight_layout()
    plt.show()

## 7. クロス集計と異常値診断

In [ ]:
if analysis_features is not None:
    cross_tables = {
        'admin_process × ai_usecase': pd.crosstab(exploded['admin_process'], exploded['ai_usecase']),
        'policy_domain × admin_process': pd.crosstab(plot_frame['policy_domain'], plot_frame['admin_process']),
        'policy_domain × ai_applicability': pd.crosstab(plot_frame['policy_domain'], plot_frame['ai_applicability']),
    }
    for title, table in cross_tables.items():
        display(table)
        plt.figure(figsize=(max(9, table.shape[1] * 0.8), max(6, table.shape[0] * 0.5)))
        sns.heatmap(table, cmap='Blues', annot=True, fmt='g')
        plt.title(title)
        plt.tight_layout()
        plt.show()

    diagnostics = pd.Series({
        'rows': len(plot_frame),
        'U99_rate': plot_frame['ai_usecase'].map(decode_ai_usecase).map(lambda values: values == ['U99']).mean(),
        'ai_applicability_0_rate': plot_frame['ai_applicability'].eq(0).mean(),
        'api_retry_total': plot_frame['api_retry_count'].sum(),
        'validation_retry_total': plot_frame['validation_retry_count'].sum(),
        'validation_error_rows_current_run': 0 if full_result is None else len(full_result['errors']),
    })
    display(diagnostics.to_frame('value'))